**Finite Differences versus Symbolic Regression**

*Mini-project by Hugo Robijns inspired by the work of Miles Cranmer and Liz Tan (PySR/SymTorch), and the principles of PhysicsX (Neural Surrogates/LPMs as a way of transforming classical numerical engineering processes).*

PhysicsX seeks to revolutionise engineering through the use of Large Physics Models, foundational engineering models trained on huge amounts of simulation and real-world data which can be used to provide inference *many orders of magnitudes faster* than equivalent numerical processes (FEM etc.). This allows real-time simulation and more thorough exploration of the design space.

Alongside this huge speed-up advantage:
-  this approach is flexible (e.g. to different geometries/boundary conditions) instead of having to carry out a completely new numerical analysis each time (big cost advantage, only need to train these models once, although this training is large and expensive).
- functions much better in high dimensional work.

Some caveats/downsides/warning points are:
- this approach is strongly tied to its training data and these LPMs could therefore be biased (they're not inherently grounded to the maths in the way a numerical PDE solver is).
- large costs involved with improving/retraining this model if this is a huge computational job each time.
- although generalisation should be good, if it's far outside its training data scope, it might struggle where a numerical method will still be able to proceed (important for chaotic and non-linear systems).
- the models are black boxes, and therefore far less interpretable.

This notebook explores the final point, using recent work from the group where I'm carrying out my Master's project (https://astroautomata.com/), symbolic distillation of neural networks, and applying it to these neural surrogates to shed light on what is actually being 'learnt'.
___

Importing the relevant libraries (bit fiddly with Julia dependancies...)

In [ ]:
!pip install pysr
!pip install git+https://github.com/astroautomata/SymTorch.git

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import pysr
import symtorch

___
**Data Generation: FD solver**

We solve Poisson's equation (applications: gravity, electrostatics, thermodynamics, fluids etc.) using finite differences. This pedagogic example could be extended to a more complicated geometry/mesh or a different PDE.

In [3]:
def solve_poisson_fd(n=32, iters=300):
    h = 1.0 / (n - 1)

    x = np.linspace(0, 1, n)
    y = np.linspace(0, 1, n)

    f = np.random.randn(n, n)
    u = np.zeros((n, n))

    for _ in range(iters):
        u_new = u.copy()
        u_new[1:-1,1:-1] = 0.25 * (
            u[2:,1:-1] + u[:-2,1:-1] +
            u[1:-1,2:] + u[1:-1,:-2] -
            h**2 * f[1:-1,1:-1]
        )
        u = u_new

    return x, y, f, u

In [4]:
def generate_dataset(num_samples=30, n=32):
    X_list, Y_list = [], []

    for _ in range(num_samples):
        x, y, f, u = solve_poisson_fd(n)

        Xg, Yg = np.meshgrid(x, y, indexing="ij")

        X = np.stack([
            Xg.flatten(),
            Yg.flatten(),
            f.flatten()
        ], axis=1)

        Y = u.flatten()[:, None]

        X_list.append(X)
        Y_list.append(Y)

    return np.vstack(X_list), np.vstack(Y_list)

X, Y = generate_dataset(30)
print(X.shape, Y.shape)

(30720, 3) (30720, 1)


___
**Training a Neural Surrogate**

We use the simulated data to train a simple MLP.

In [5]:
X_t = torch.tensor(X, dtype=torch.float32)
Y_t = torch.tensor(Y, dtype=torch.float32)

model = nn.Sequential(
    torch.nn.Linear(3, 64),
    torch.nn.Tanh(),
    torch.nn.Linear(64, 64),
    torch.nn.Tanh(),
    torch.nn.Linear(64, 1)
)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(301):
    pred = model(X_t)
    loss = ((pred - Y_t)**2).mean()

    opt.zero_grad()
    loss.backward()
    opt.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss {loss.item():.6f}")

Epoch 0, Loss 0.006207
Epoch 50, Loss 0.000053
Epoch 100, Loss 0.000024
Epoch 150, Loss 0.000014
Epoch 200, Loss 0.000010
Epoch 250, Loss 0.000007
Epoch 300, Loss 0.000006


___
**Symbolic Regression**


In [6]:
def model_fn(x_numpy):
    x = torch.tensor(x_numpy, dtype=torch.float32)
    with torch.no_grad():
        y = model(x).numpy()
    return y

sym_model = symtorch.SymbolicModel(model_fn)

sr_params_1 = {
    'niterations': 10}

sym_model.distill(
    X,
    sr_params=sr_params_1
)

/usr/local/lib/python3.12/dist-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pysr/sr.py:2265: UserWarning: Note: you are running with more than 10,000 datapoints. You should consider turning on batching (https://ai.damtp.cam.ac.uk/pysr/options/#batching). You should also reconsider if you need that many datapoints. Unless you have a large amount of noise (in which case you should smooth your dataset first), generally < 10,000 datapoints is enough to find a functional form with symbolic regression. More datapoints will lower the search speed.
  warnings.warn(
Compiling Julia backend...
INFO:pysr.sr:Compiling Julia backend...
[ Info: Note: you are running with more than 10,000 datapoints. You should consider turning on batching (`options.batching`), and also if you need that many datapoints. Unless you have a large amount of noise (in which case you shoul


Expressions evaluated per second: 0.000e+00
Progress: 0 / 310 total iterations (0.000%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
───────────────────────────────────────────────────────────────────────────────────────────────────
════════════════════════════════════════════════════════════════════════════════════════════════════
Press 'q' and then <enter> to stop execution early.

Expressions evaluated per second: 6.220e+01
Progress: 2 / 310 total iterations (0.645%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
3           4.243e-05  0.000e+00  y = -0.078497 * -0.078497
4           3.127e-06

[ Info: Final population:
[ Info: Results saved to:


{0: PySRRegressor.equations_ = [
 	    pick     score                                           equation  \
 	0         0.000000                                     -0.00012320734   
 	1   >>>>  0.055313                                 x2 * -0.0005678893   
 	2         0.020405                           x2 * (x1 * -0.001129538)   
 	3         0.015642                   x1 * ((x2 * -0.0015676373) * x1)   
 	4         0.007057            (x1 * x1) * (x1 * (x2 * -0.0018941185))   
 	5         0.013004  (x2 + 0.34344855) * (((x1 * x1) * -0.002119618...   
 	6         0.012562  ((x1 * -0.0021196199) * (x1 * (x2 + x0))) * (x...   
 	7         0.019038  (x1 * -0.002824049) * ((x2 + x0) * (((x0 * -0....   
 	8         0.014555  ((x1 * (x1 * -0.0034407894)) * (x2 + x0)) * ((...   
 	9         0.046645  (((x1 + (x0 * -1.4150252)) * 3.144649) + x0) *...   
 	10        0.006547  x2 * sin((-0.0056297043 * ((x0 + -0.6280589) *...   
 	11        0.006180  x2 * ((((x0 + -0.60404295) * -0.002730914) * 

  - SR_output/block_138290115531632/dim0_1774948549/hall_of_fame.csv
